# 05 - Yogyakarta LSTM Autoencoder Sensitivity Analysis

This notebook compares LSTM Autoencoder anomaly detection behavior across different sequence windows and feature groups for DI Yogyakarta weather data.

Experiments:

- 7-day versus 30-day sequence windows
- Rainfall-only features
- Wind-only features
- All weather features

This notebook is a sensitivity experiment. It does not replace the baseline model from notebook 02 and does not overwrite baseline reports from notebook 03.

## 1. Setup

In [ ]:
from pathlib import Path
import json
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler
from IPython.display import Markdown, display

try:
    import tensorflow as tf
    from tensorflow.keras import Model
    from tensorflow.keras.callbacks import EarlyStopping
    from tensorflow.keras.layers import Dense, Dropout, Input, LSTM, RepeatVector, TimeDistributed
    from tensorflow.keras.optimizers import Adam
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "TensorFlow is not installed in this Jupyter kernel. Install requirements-notebook.txt first, "
        "then restart the kernel and rerun this notebook."
    ) from exc

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path("..").resolve()
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "yogyakarta_weather_features.csv"
REPORT_DIR = PROJECT_ROOT / "reports"
SENSITIVITY_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "sensitivity"
BASELINE_ALERTS_PATH = REPORT_DIR / "alerts.csv"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
SENSITIVITY_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MAX_EPOCHS = 50
BATCH_SIZE = 32
PATIENCE = 8
LEARNING_RATE = 0.001

print("Project root:", PROJECT_ROOT)
print("Processed data:", PROCESSED_PATH)
print("Reports:", REPORT_DIR)

## 2. Load Processed Features

In [ ]:
if not PROCESSED_PATH.exists():
    raise FileNotFoundError(
        "Run notebook 01 first: data/processed/yogyakarta_weather_features.csv was not found."
    )

data = pd.read_csv(PROCESSED_PATH, parse_dates=["date"], dtype={"station_id": "string"})
data = data.sort_values(["station_id", "date"]).reset_index(drop=True)

print("Rows:", len(data))
print("Date range:", data["date"].min(), "to", data["date"].max())
display(data.groupby("station_id")["date"].agg(["min", "max", "count"]))
display(data.head())

## 3. Experiment Configurations

In [ ]:
EXPERIMENTS = [
    {"experiment_id": "rain_7d", "sequence_length": 7, "feature_group": "rain_only"},
    {"experiment_id": "wind_7d", "sequence_length": 7, "feature_group": "wind_only"},
    {"experiment_id": "all_7d", "sequence_length": 7, "feature_group": "all_features"},
    {"experiment_id": "rain_30d", "sequence_length": 30, "feature_group": "rain_only"},
    {"experiment_id": "wind_30d", "sequence_length": 30, "feature_group": "wind_only"},
    {"experiment_id": "all_30d", "sequence_length": 30, "feature_group": "all_features"},
]

pd.DataFrame(EXPERIMENTS)

## 4. Helper Functions

In [ ]:
def resolve_feature_groups(data: pd.DataFrame) -> dict:
    all_features = [
        "Tn",
        "Tx",
        "Tavg",
        "RH_avg",
        "RR",
        "ss",
        "ff_x",
        "ff_avg",
        "temp_range",
        "rain_3d",
        "rain_7d",
        "rain_change_1d",
        "wind_change_1d",
        "ddd_x_sin",
        "ddd_x_cos",
        "day_of_year_sin",
        "day_of_year_cos",
        "station_96855",
        "missing_RR",
        "missing_ff_x",
        "missing_ff_avg",
        "missing_RH_avg",
    ]

    return {
        "rain_only": ["RR", "rain_3d", "rain_7d", "rain_change_1d", "missing_RR"],
        "wind_only": [
            "ff_x",
            "ff_avg",
            "wind_change_1d",
            "ddd_x_sin",
            "ddd_x_cos",
            "missing_ff_x",
            "missing_ff_avg",
        ],
        "all_features": all_features,
    }


def validate_required_columns(data: pd.DataFrame, feature_names: list, experiment_id: str) -> None:
    required_metadata = ["date", "station_id", "station_name", "region_name", "RR", "rain_3d", "rain_7d", "ff_x", "ff_avg"]
    missing = [column for column in required_metadata + feature_names if column not in data.columns]
    if missing:
        raise ValueError(f"{experiment_id} cannot run because these columns are missing: {missing}")


def chronological_split(data: pd.DataFrame) -> tuple:
    ordered_dates = np.array(sorted(data["date"].dropna().unique()))
    if len(ordered_dates) < 30:
        raise ValueError("Not enough unique dates for chronological train/validation/test split.")

    train_end = int(len(ordered_dates) * 0.70)
    validation_end = int(len(ordered_dates) * 0.85)

    train_dates = ordered_dates[:train_end]
    validation_dates = ordered_dates[train_end:validation_end]
    test_dates = ordered_dates[validation_end:]

    train_df = data[data["date"].isin(train_dates)].copy()
    validation_df = data[data["date"].isin(validation_dates)].copy()
    test_df = data[data["date"].isin(test_dates)].copy()

    return train_df, validation_df, test_df


def fit_transform_splits(train_df, validation_df, test_df, feature_names) -> tuple:
    scaler = RobustScaler()
    scaler.fit(train_df[feature_names])

    def scale_frame(df):
        scaled = df.copy()
        scaled[feature_names] = scaler.transform(scaled[feature_names])
        return scaled

    return scale_frame(train_df), scale_frame(validation_df), scale_frame(test_df), scaler


def build_sequences(feature_df, metadata_df, feature_names, sequence_length) -> tuple:
    x_values = []
    metadata_rows = []
    skipped_windows = 0

    feature_df = feature_df.sort_values(["station_id", "date"]).reset_index(drop=True)
    metadata_df = metadata_df.sort_values(["station_id", "date"]).reset_index(drop=True)

    for station_id, meta_group in metadata_df.groupby("station_id", sort=False):
        group_index = meta_group.index
        g_meta = meta_group.reset_index(drop=True)
        g_feature = feature_df.loc[group_index].reset_index(drop=True)
        values = g_feature[feature_names].to_numpy(dtype=np.float32)

        for end_idx in range(sequence_length - 1, len(g_meta)):
            start_idx = end_idx - sequence_length + 1
            window_dates = g_meta.loc[start_idx:end_idx, "date"]
            day_differences = window_dates.diff().dt.days.iloc[1:]

            if not (day_differences == 1).all():
                skipped_windows += 1
                continue

            x_values.append(values[start_idx:end_idx + 1])
            end_row = g_meta.loc[end_idx]
            metadata_rows.append(
                {
                    "date": end_row["date"],
                    "station_id": str(end_row["station_id"]),
                    "station_name": end_row.get("station_name", ""),
                    "region_name": end_row.get("region_name", ""),
                    "province_name": end_row.get("province_name", ""),
                    "RR": float(end_row["RR"]),
                    "rain_3d": float(end_row["rain_3d"]),
                    "rain_7d": float(end_row["rain_7d"]),
                    "ff_x": float(end_row["ff_x"]),
                    "ff_avg": float(end_row["ff_avg"]),
                }
            )

    if not x_values:
        empty_shape = (0, sequence_length, len(feature_names))
        return np.empty(empty_shape, dtype=np.float32), pd.DataFrame(metadata_rows), skipped_windows

    return np.stack(x_values).astype(np.float32), pd.DataFrame(metadata_rows), skipped_windows


def build_compact_lstm_autoencoder(sequence_length, n_features, learning_rate=0.001) -> Model:
    inputs = Input(shape=(sequence_length, n_features))
    encoded = LSTM(32, activation="tanh", return_sequences=True)(inputs)
    encoded = Dropout(0.15)(encoded)
    encoded = LSTM(16, activation="tanh", return_sequences=False)(encoded)

    decoded = RepeatVector(sequence_length)(encoded)
    decoded = LSTM(16, activation="tanh", return_sequences=True)(decoded)
    decoded = Dropout(0.15)(decoded)
    decoded = LSTM(32, activation="tanh", return_sequences=True)(decoded)
    outputs = TimeDistributed(Dense(n_features))(decoded)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss="mse")
    return model


def weight_vector(feature_names: list, feature_group: str) -> np.ndarray:
    baseline_weights = {
        "RR": 3.0,
        "rain_3d": 3.0,
        "rain_7d": 3.0,
        "rain_change_1d": 2.0,
        "ff_x": 2.5,
        "ff_avg": 2.5,
        "wind_change_1d": 2.5,
        "RH_avg": 1.5,
        "Tavg": 1.0,
        "Tn": 1.0,
        "Tx": 1.0,
        "temp_range": 1.0,
        "ss": 1.0,
        "ddd_x_sin": 0.8,
        "ddd_x_cos": 0.8,
        "day_of_year_sin": 0.5,
        "day_of_year_cos": 0.5,
        "station_96855": 0.5,
        "missing_RR": 0.5,
        "missing_ff_x": 0.5,
        "missing_ff_avg": 0.5,
        "missing_RH_avg": 0.5,
    }

    rain_weights = {"RR": 3.0, "rain_3d": 3.0, "rain_7d": 3.0, "rain_change_1d": 2.0, "missing_RR": 0.5}
    wind_weights = {
        "ff_x": 3.0,
        "ff_avg": 3.0,
        "wind_change_1d": 3.0,
        "ddd_x_sin": 1.0,
        "ddd_x_cos": 1.0,
        "missing_ff_x": 0.5,
        "missing_ff_avg": 0.5,
    }

    if feature_group == "rain_only":
        source = rain_weights
    elif feature_group == "wind_only":
        source = wind_weights
    else:
        source = baseline_weights

    weights = np.array([source.get(name, 1.0) for name in feature_names], dtype=np.float32)
    return weights / weights.mean()


def weighted_reconstruction_error(x_true, x_pred, weights) -> np.ndarray:
    weights = weights.reshape(1, 1, -1)
    squared_error = np.square(x_true - x_pred)
    return np.mean(squared_error * weights, axis=(1, 2))


def status_from_score(score, thresholds) -> str:
    if score > thresholds["p995"]:
        return "AWAS"
    if score > thresholds["p99"]:
        return "SIAGA"
    if score > thresholds["p95"]:
        return "WASPADA"
    return "NORMAL"


def derive_physical_thresholds(training_df: pd.DataFrame) -> dict:
    return {
        "rain_daily_threshold": float(training_df["RR"].quantile(0.95)),
        "rain_3d_threshold": float(training_df["rain_3d"].quantile(0.95)),
        "wind_max_threshold": float(training_df["ff_x"].quantile(0.95)),
        "wind_avg_threshold": float(training_df["ff_avg"].quantile(0.95)),
    }


def classify_alert(row: pd.Series, physical_thresholds: dict) -> tuple:
    rain_alert = bool(
        row["RR"] >= physical_thresholds["rain_daily_threshold"]
        or row["rain_3d"] >= physical_thresholds["rain_3d_threshold"]
    )
    wind_alert = bool(
        row["ff_x"] >= physical_thresholds["wind_max_threshold"]
        or row["ff_avg"] >= physical_thresholds["wind_avg_threshold"]
    )

    if row["status"] == "NORMAL":
        return "NORMAL", ""
    if rain_alert and wind_alert:
        alert_type = "CURAH_HUJAN_EKSTREM_DAN_ANGIN_KENCANG"
    elif rain_alert:
        alert_type = "CURAH_HUJAN_EKSTREM"
    elif wind_alert:
        alert_type = "ANGIN_KENCANG"
    else:
        alert_type = "ANOMALI_CUACA"

    triggers = [f"Anomaly score above {row['status']} threshold"]
    if rain_alert:
        triggers.append("Rainfall above local percentile threshold")
    if wind_alert:
        triggers.append("Wind above local percentile threshold")

    return alert_type, "; ".join(triggers)


feature_groups = resolve_feature_groups(data)
for config in EXPERIMENTS:
    validate_required_columns(data, feature_groups[config["feature_group"]], config["experiment_id"])

print("Feature groups are ready:")
for name, columns in feature_groups.items():
    print(f"{name}: {len(columns)} features")

## 5. Run Sensitivity Experiments

This cell trains six compact LSTM Autoencoder models. It can take several minutes depending on the server.

In [ ]:
train_df, validation_df, test_df = chronological_split(data)
physical_thresholds = derive_physical_thresholds(train_df)

summary_rows = []
alert_count_rows = []
top_anomaly_frames = []
score_frames = []

for config in EXPERIMENTS:
    experiment_id = config["experiment_id"]
    sequence_length = config["sequence_length"]
    feature_group = config["feature_group"]
    feature_names = feature_groups[feature_group]

    print("=" * 80)
    print(f"Running {experiment_id}: {sequence_length} days, {feature_group}, {len(feature_names)} features")

    tf.keras.backend.clear_session()

    train_scaled, validation_scaled, test_scaled, scaler = fit_transform_splits(
        train_df, validation_df, test_df, feature_names
    )

    eval_df = pd.concat([validation_df, test_df], ignore_index=True)
    eval_scaled = pd.concat([validation_scaled, test_scaled], ignore_index=True)

    x_train, train_meta, skipped_train = build_sequences(train_scaled, train_df, feature_names, sequence_length)
    x_validation, validation_meta, skipped_validation = build_sequences(
        validation_scaled, validation_df, feature_names, sequence_length
    )
    x_test, test_meta, skipped_test = build_sequences(test_scaled, test_df, feature_names, sequence_length)
    x_eval, eval_meta, skipped_eval = build_sequences(eval_scaled, eval_df, feature_names, sequence_length)

    if len(x_train) == 0 or len(x_validation) == 0 or len(x_eval) == 0:
        raise ValueError(
            f"{experiment_id} has too few consecutive rows for sequence_length={sequence_length}. "
            f"Shapes: train={x_train.shape}, validation={x_validation.shape}, eval={x_eval.shape}"
        )

    model = build_compact_lstm_autoencoder(sequence_length, len(feature_names), LEARNING_RATE)
    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=PATIENCE,
        restore_best_weights=True,
        mode="min",
    )

    history = model.fit(
        x_train,
        x_train,
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(x_validation, x_validation),
        callbacks=[early_stopping],
        verbose=1,
    )

    weights = weight_vector(feature_names, feature_group)
    validation_pred = model.predict(x_validation, verbose=0)
    validation_scores = weighted_reconstruction_error(x_validation, validation_pred, weights)
    thresholds = {
        "p95": float(np.percentile(validation_scores, 95)),
        "p99": float(np.percentile(validation_scores, 99)),
        "p995": float(np.percentile(validation_scores, 99.5)),
    }

    eval_pred = model.predict(x_eval, verbose=0)
    eval_scores = weighted_reconstruction_error(x_eval, eval_pred, weights)
    eval_meta = eval_meta.copy()
    eval_meta["experiment_id"] = experiment_id
    eval_meta["sequence_length"] = sequence_length
    eval_meta["feature_group"] = feature_group
    eval_meta["anomaly_score"] = eval_scores
    eval_meta["status"] = [status_from_score(float(score), thresholds) for score in eval_scores]

    alert_values = eval_meta.apply(lambda row: classify_alert(row, physical_thresholds), axis=1)
    eval_meta["alert_type"] = [value[0] for value in alert_values]
    eval_meta["triggers"] = [value[1] for value in alert_values]
    eval_meta["threshold_p95"] = thresholds["p95"]
    eval_meta["threshold_p99"] = thresholds["p99"]
    eval_meta["threshold_p995"] = thresholds["p995"]

    status_counts = eval_meta["status"].value_counts().reindex(["NORMAL", "WASPADA", "SIAGA", "AWAS"], fill_value=0)
    alert_type_counts = eval_meta["alert_type"].value_counts()

    for status, count in status_counts.items():
        alert_count_rows.append(
            {
                "experiment_id": experiment_id,
                "sequence_length": sequence_length,
                "feature_group": feature_group,
                "count_type": "status",
                "label": status,
                "count": int(count),
            }
        )

    for alert_type, count in alert_type_counts.items():
        alert_count_rows.append(
            {
                "experiment_id": experiment_id,
                "sequence_length": sequence_length,
                "feature_group": feature_group,
                "count_type": "alert_type",
                "label": alert_type,
                "count": int(count),
            }
        )

    non_normal = eval_meta[eval_meta["status"] != "NORMAL"].copy()
    top_anomalies = non_normal.sort_values("anomaly_score", ascending=False).head(15)
    top_anomaly_frames.append(top_anomalies)
    score_frames.append(eval_meta[["experiment_id", "sequence_length", "feature_group", "date", "station_id", "anomaly_score", "status", "alert_type"]])

    top_row = eval_meta.sort_values("anomaly_score", ascending=False).iloc[0]
    summary_rows.append(
        {
            "experiment_id": experiment_id,
            "sequence_length": sequence_length,
            "feature_group": feature_group,
            "n_features": len(feature_names),
            "train_sequences": int(len(x_train)),
            "validation_sequences": int(len(x_validation)),
            "test_sequences": int(len(x_test)),
            "eval_sequences": int(len(x_eval)),
            "skipped_train_windows": int(skipped_train),
            "skipped_validation_windows": int(skipped_validation),
            "skipped_test_windows": int(skipped_test),
            "skipped_eval_windows": int(skipped_eval),
            "best_val_loss": float(np.min(history.history["val_loss"])),
            "stopped_epoch": int(len(history.history["loss"])),
            "threshold_p95": thresholds["p95"],
            "threshold_p99": thresholds["p99"],
            "threshold_p995": thresholds["p995"],
            "normal_count": int(status_counts["NORMAL"]),
            "waspada_count": int(status_counts["WASPADA"]),
            "siaga_count": int(status_counts["SIAGA"]),
            "awas_count": int(status_counts["AWAS"]),
            "non_normal_count": int(len(non_normal)),
            "top_date": pd.Timestamp(top_row["date"]).date().isoformat(),
            "top_station_id": str(top_row["station_id"]),
            "top_alert_type": top_row["alert_type"],
            "top_anomaly_score": float(top_row["anomaly_score"]),
        }
    )

    artifact_payload = {
        "experiment": config,
        "feature_names": feature_names,
        "thresholds": thresholds,
        "history": {key: [float(value) for value in values] for key, values in history.history.items()},
    }
    (SENSITIVITY_ARTIFACT_DIR / f"{experiment_id}_summary.json").write_text(
        json.dumps(artifact_payload, indent=2), encoding="utf-8"
    )

summary_df = pd.DataFrame(summary_rows)
alert_counts_df = pd.DataFrame(alert_count_rows)
top_anomalies_df = pd.concat(top_anomaly_frames, ignore_index=True) if top_anomaly_frames else pd.DataFrame()
scores_df = pd.concat(score_frames, ignore_index=True) if score_frames else pd.DataFrame()

display(summary_df)
display(alert_counts_df.head(20))
display(top_anomalies_df.head(20))

## 6. Export Reports

In [ ]:
if BASELINE_ALERTS_PATH.exists():
    baseline_alerts = pd.read_csv(BASELINE_ALERTS_PATH, parse_dates=["date"], dtype={"station_id": "string"})
    baseline_non_normal = baseline_alerts[baseline_alerts["status"] != "NORMAL"].copy()
    baseline_keys = set(
        baseline_non_normal["station_id"].astype(str) + "|" + baseline_non_normal["date"].dt.date.astype(str)
    )

    overlap_rows = []
    for experiment_id, group in scores_df[scores_df["status"] != "NORMAL"].groupby("experiment_id"):
        experiment_keys = set(group["station_id"].astype(str) + "|" + group["date"].dt.date.astype(str))
        overlap_count = len(experiment_keys & baseline_keys)
        overlap_rows.append(
            {
                "experiment_id": experiment_id,
                "baseline_non_normal_dates": len(baseline_keys),
                "experiment_non_normal_dates": len(experiment_keys),
                "overlap_count": overlap_count,
                "overlap_pct_of_experiment": overlap_count / len(experiment_keys) if experiment_keys else 0.0,
                "overlap_pct_of_baseline": overlap_count / len(baseline_keys) if baseline_keys else 0.0,
            }
        )

    overlap_df = pd.DataFrame(overlap_rows)
    summary_df = summary_df.merge(overlap_df, on="experiment_id", how="left")
else:
    overlap_df = pd.DataFrame()
    print("Baseline alerts.csv was not found. Baseline overlap comparison is skipped.")

summary_path = REPORT_DIR / "sensitivity_experiment_summary.csv"
alert_counts_path = REPORT_DIR / "sensitivity_alert_counts.csv"
top_anomalies_path = REPORT_DIR / "sensitivity_top_anomalies.csv"

summary_df.to_csv(summary_path, index=False)
alert_counts_df.to_csv(alert_counts_path, index=False)
top_anomalies_df.to_csv(top_anomalies_path, index=False)

print("Saved:", summary_path)
print("Saved:", alert_counts_path)
print("Saved:", top_anomalies_path)

display(summary_df)
if not overlap_df.empty:
    display(overlap_df)

## 7. Visual Comparison

In [ ]:
status_plot_df = alert_counts_df[alert_counts_df["count_type"] == "status"].copy()
status_plot_df["status"] = pd.Categorical(
    status_plot_df["label"], categories=["NORMAL", "WASPADA", "SIAGA", "AWAS"], ordered=True
)

plt.figure(figsize=(12, 6))
sns.barplot(data=status_plot_df, x="experiment_id", y="count", hue="status")
plt.title("Sensitivity Analysis - Warning Status Counts")
plt.xlabel("Experiment")
plt.ylabel("Window count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
status_plot_path = REPORT_DIR / "sensitivity_status_comparison.png"
plt.savefig(status_plot_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", status_plot_path)

plt.figure(figsize=(12, 6))
sns.boxplot(data=scores_df, x="experiment_id", y="anomaly_score")
plt.title("Sensitivity Analysis - Anomaly Score Distribution")
plt.xlabel("Experiment")
plt.ylabel("Anomaly score")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
score_plot_path = REPORT_DIR / "sensitivity_score_distribution.png"
plt.savefig(score_plot_path, dpi=160, bbox_inches="tight")
plt.show()
print("Saved:", score_plot_path)

## 8. Lecturer-Facing Interpretation

In [ ]:
summary_display_columns = [
    "experiment_id",
    "sequence_length",
    "feature_group",
    "n_features",
    "best_val_loss",
    "threshold_p95",
    "non_normal_count",
    "waspada_count",
    "siaga_count",
    "awas_count",
    "top_date",
    "top_alert_type",
    "top_anomaly_score",
]
display(summary_df[summary_display_columns].sort_values(["sequence_length", "feature_group"]))

message = """
### Ringkasan interpretasi

Notebook ini bukan mencari akurasi klasifikasi bencana, karena data yang dipakai belum punya label kejadian bencana yang tervalidasi. Tujuannya adalah melihat seberapa berubah hasil deteksi anomali LSTM Autoencoder ketika lebar window dan kelompok fitur diganti.

- Model 7 hari membaca pola yang lebih pendek, jadi biasanya lebih responsif terhadap perubahan cuaca yang mendadak.
- Model 30 hari membaca konteks yang lebih panjang, jadi biasanya lebih stabil dan tidak terlalu mudah bereaksi terhadap satu-dua hari yang ekstrem.
- Rain-only berguna untuk melihat apakah pola hujan saja sudah cukup menjelaskan anomali curah hujan.
- Wind-only berguna untuk melihat apakah pola angin saja sudah cukup menjelaskan anomali angin kencang.
- All-features tetap kandidat utama untuk model utama karena membaca konteks cuaca paling lengkap: suhu, kelembapan, hujan, angin, arah angin, musim, dan informasi stasiun.

Rekomendasi awal: gunakan hasil all-features sebagai kandidat utama, lalu pakai rain-only dan wind-only sebagai pembanding untuk menjelaskan apakah anomali tersebut lebih kuat dari sisi hujan, angin, atau kombinasi pola cuaca.
"""

display(Markdown(message))

## 9. Verification

In [ ]:
expected_paths = [
    REPORT_DIR / "sensitivity_experiment_summary.csv",
    REPORT_DIR / "sensitivity_alert_counts.csv",
    REPORT_DIR / "sensitivity_top_anomalies.csv",
    REPORT_DIR / "sensitivity_status_comparison.png",
    REPORT_DIR / "sensitivity_score_distribution.png",
]

verification = pd.DataFrame(
    {
        "path": [str(path.relative_to(PROJECT_ROOT)) for path in expected_paths],
        "exists": [path.exists() for path in expected_paths],
    }
)

display(verification)

if len(summary_df) != len(EXPERIMENTS):
    raise AssertionError(f"Expected {len(EXPERIMENTS)} experiment rows, found {len(summary_df)}.")

missing_outputs = verification[~verification["exists"]]
if not missing_outputs.empty:
    raise AssertionError("Some sensitivity outputs were not created.")

print("Sensitivity notebook verification passed.")